# Hamiltonian Adaptive Ternary Tree

HATT is an algorithm for the construction of Ternary-Tree encodings which are optimised for the Hamiltonian of interest.

This notebook shows how to reproduce the results of: 
Y. Liu et al., "HATT: Hamiltonian Adaptive Ternary Tree for Optimizing Fermion-to-Qubit Mapping," 2025 IEEE International Symposium on High Performance Computer Architecture (HPCA), Las Vegas, NV, USA, 2025, pp. 143-157, doi: 10.1109/HPCA61900.2025.00022.

In [1]:
import pickle
from ferrmion.encode import TernaryTree
from pathlib import Path
from ferrmion.encode import TernaryTree
from ferrmion.encode.ternary_tree_node import TTNode
from pathlib import Path
import numpy as np

def water_integrals():
    folder = Path.cwd().joinpath(Path("../../../python/tests/"))
    with open(folder.joinpath("./data/water_1e.pkl"), 'rb') as file:
        ones = pickle.load(file)

    with open(folder.joinpath("./data/water_2e.pkl"), 'rb') as file:
        twos = pickle.load(file)
    return (ones, twos)

ones, twos = water_integrals()
# to avoid double-counting
twos = 0.5* twos

# Preprocess Hamiltonian

In [2]:
from itertools import product
import ferrmion as fr
def signature_char_to_ipowers(char:str):
    match char:
        case "+":
            return [1,-1j]
        case "-":
            return [1,1j]
        case "_":
            raise ValueError("Signature must contain only + and -")

def hamiltonian_term_to_majorana(majorana_ham, coeffs, signature):
    assert len(signature) == coeffs.ndim
    non_zero = np.where(coeffs != 0)
    non_zero_ones = [(*indices, coeffs[indices]) for indices in zip(*non_zero)]
    normalisation = 0.5**len(signature)

    
    ipowers = np.array([signature_char_to_ipowers(c) for c in signature])
    for *inds, coeff in non_zero_ones:
        # we need two majoranas for each fermionic operator
        left_right_indices = [[0,1]]*len(signature) 
        for left_right in product(*left_right_indices):
            majorana_ind = tuple([i+lr for i,lr in zip(inds, left_right)])
            term_ipowers = np.prod([ipow[lr] for ipow,lr in zip(ipowers, left_right)])
            majorana_ham[majorana_ind] = majorana_ham.get(majorana_ind, 0)
            majorana_ham[majorana_ind] += normalisation * coeff * term_ipowers

    return majorana_ham

def fermionic_to_majorana(hamiltonian_terms:list[tuple[np.ndarray, str]])-> dict[tuple[int],np.complex64]:
    total_ham = {}
    for coeffs, signature in hamiltonian_terms:
        total_ham.update(hamiltonian_term_to_majorana(total_ham, coeffs=coeffs, signature=signature))
    return total_ham

majorana_ham = fermionic_to_majorana([(ones, "+-"),(twos,"++--")])

In [3]:
import ferrmion as fr
root_node = TTNode()
root_node.branch_majorana_map

{'x': None, 'z': None, 'y': None}

In [79]:
tt = fr.TernaryTree(n_modes=1,root_node=TTNode(qubit_label=0))
tt.enumeration_scheme = tt.default_enumeration_scheme()
tt.root_node.branch_strings

{'x', 'y', 'z'}

# Algorithm 1

In [ ]:
n_modes = ones.shape[0]

nodes = {i: None for i in range(2*n_modes)}
for i in range(n_modes//2):
    parent_index = n_modes//2 + i 

    mins = sorted(weights.items(), key=lambda kv: (kv[1], kv[0]))[:3]

    parent = nodes.get(
        parent_index, TTNode(parent=None, root_path="", qubit_label=i)
    )

    for min, child_string in zip(mins, ["x", "y", "z"][: len(mins)]):
        possible_child = nodes[min[0]]
        if isinstance(possible_child, TTNode):
            parent.add_child(which_child=child_string, child_node=possible_child)

    new_weight = 0
    for index, weight in mins:
        new_weight += weight
        weights.pop(index)
        nodes.pop(index)

    nodes[parent_index] = parent
    weights[parent_index] = new_weight

assert len(nodes) == 1
root_node = [*nodes.values()][0]
